# English Premier League VAR Analysis
## Part 3: VAR Impact vs League Performance

---

### Overview

This notebook investigates the relationship between **VAR (Video Assistant Referee) decisions** and **team performance** in the English Premier League across 6 seasons (2019/2020 - 2024/2025).

### Research Questions

| # | Question | Method |
|---|----------|--------|
| 1 | Do "Big 6" clubs receive favorable VAR treatment? | Independent t-test |
| 2 | Does VAR net score correlate with final points? | Pearson/Spearman correlation |
| 3 | Does VAR affect final league position? | Regression analysis |

### Data Sources

- **VAR Statistics**: `var_decisions_all_seasons.csv` (scraped from ESPN)
- **EPL Standings**: Hardcoded historical data from official Premier League records

### Author
Adam Bahrin | [GitHub](https://github.com/adamhkb)

In [ ]:
# ===========================================================================
# IMPORTS & CONFIGURATION
# ===========================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, ttest_ind
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('✅ Libraries loaded successfully')

---

## 1. Load VAR Data

The VAR dataset contains team-level statistics for each Premier League season:

| Column | Description |
|--------|-------------|
| `team_name` | Club name |
| `net_score` | Overall VAR impact (+ve = benefited, -ve = disadvantaged) |
| `net_goal_score` | Net goals gained/lost due to VAR |
| `overturns_total` | Total VAR overturns involving the team |
| `year` | Season (e.g., "2023/2024") |

In [ ]:
# Load VAR decisions data
var_df = pd.read_csv('var_decisions_all_seasons.csv')

# Clean and convert numeric columns
var_df['net_score'] = pd.to_numeric(
    var_df['net_score'].astype(str).str.replace('+', ''), 
    errors='coerce'
)
var_df['net_goal_score'] = pd.to_numeric(
    var_df['net_goal_score'].astype(str).str.replace('+', ''), 
    errors='coerce'
).fillna(0)

print(f'✅ Loaded VAR data: {len(var_df)} team-season records')
print(f'   Seasons covered: {sorted(var_df["year"].unique())}')
print(f'   Teams per season: ~{len(var_df) // var_df["year"].nunique()}')

var_df.head()

---

## 2. Load EPL Final Standings

Historical EPL final standings are hardcoded below. This approach was chosen because:

1. **Reliability**: Historical data doesn't change
2. **No dependencies**: Avoids breaking when websites change structure
3. **Speed**: No network requests needed

> **Note**: 2024/2025 data should be updated with official final standings when available.

### Data Fields

| Field | Description |
|-------|-------------|
| `team` | Club name |
| `year` | Season |
| `position` | Final league position (1-20) |
| `points` | Total points |
| `gd` | Goal difference |

In [ ]:
# ===========================================================================
# EPL FINAL STANDINGS (2019/2020 - 2024/2025)
# Source: Official Premier League records
# ===========================================================================

epl_data = [
    # 2019/2020 - Liverpool's title-winning season (99 pts)
    {'team': 'Liverpool', 'year': '2019/2020', 'position': 1, 'points': 99, 'gd': 52},
    {'team': 'Manchester City', 'year': '2019/2020', 'position': 2, 'points': 81, 'gd': 67},
    {'team': 'Manchester United', 'year': '2019/2020', 'position': 3, 'points': 66, 'gd': 30},
    {'team': 'Chelsea', 'year': '2019/2020', 'position': 4, 'points': 66, 'gd': 15},
    {'team': 'Leicester City', 'year': '2019/2020', 'position': 5, 'points': 62, 'gd': 26},
    {'team': 'Tottenham Hotspur', 'year': '2019/2020', 'position': 6, 'points': 59, 'gd': 14},
    {'team': 'Wolverhampton', 'year': '2019/2020', 'position': 7, 'points': 59, 'gd': 11},
    {'team': 'Arsenal', 'year': '2019/2020', 'position': 8, 'points': 56, 'gd': 8},
    {'team': 'Sheffield United', 'year': '2019/2020', 'position': 9, 'points': 54, 'gd': 0},
    {'team': 'Burnley', 'year': '2019/2020', 'position': 10, 'points': 54, 'gd': -7},
    {'team': 'Southampton', 'year': '2019/2020', 'position': 11, 'points': 52, 'gd': -9},
    {'team': 'Everton', 'year': '2019/2020', 'position': 12, 'points': 49, 'gd': -12},
    {'team': 'Newcastle United', 'year': '2019/2020', 'position': 13, 'points': 44, 'gd': -20},
    {'team': 'Crystal Palace', 'year': '2019/2020', 'position': 14, 'points': 43, 'gd': -19},
    {'team': 'Brighton & Hove Albion', 'year': '2019/2020', 'position': 15, 'points': 41, 'gd': -15},
    {'team': 'West Ham United', 'year': '2019/2020', 'position': 16, 'points': 39, 'gd': -13},
    {'team': 'Aston Villa', 'year': '2019/2020', 'position': 17, 'points': 35, 'gd': -26},
    {'team': 'AFC Bournemouth', 'year': '2019/2020', 'position': 18, 'points': 34, 'gd': -25},
    {'team': 'Watford', 'year': '2019/2020', 'position': 19, 'points': 34, 'gd': -28},
    {'team': 'Norwich City', 'year': '2019/2020', 'position': 20, 'points': 21, 'gd': -49},
    
    # 2020/2021 - Man City's dominant title win
    {'team': 'Manchester City', 'year': '2020/2021', 'position': 1, 'points': 86, 'gd': 51},
    {'team': 'Manchester United', 'year': '2020/2021', 'position': 2, 'points': 74, 'gd': 29},
    {'team': 'Liverpool', 'year': '2020/2021', 'position': 3, 'points': 69, 'gd': 26},
    {'team': 'Chelsea', 'year': '2020/2021', 'position': 4, 'points': 67, 'gd': 22},
    {'team': 'Leicester City', 'year': '2020/2021', 'position': 5, 'points': 66, 'gd': 18},
    {'team': 'West Ham United', 'year': '2020/2021', 'position': 6, 'points': 65, 'gd': 15},
    {'team': 'Tottenham Hotspur', 'year': '2020/2021', 'position': 7, 'points': 62, 'gd': 23},
    {'team': 'Arsenal', 'year': '2020/2021', 'position': 8, 'points': 61, 'gd': 16},
    {'team': 'Leeds United', 'year': '2020/2021', 'position': 9, 'points': 59, 'gd': 8},
    {'team': 'Everton', 'year': '2020/2021', 'position': 10, 'points': 59, 'gd': -1},
    {'team': 'Aston Villa', 'year': '2020/2021', 'position': 11, 'points': 55, 'gd': 9},
    {'team': 'Newcastle United', 'year': '2020/2021', 'position': 12, 'points': 45, 'gd': -16},
    {'team': 'Wolverhampton', 'year': '2020/2021', 'position': 13, 'points': 45, 'gd': -16},
    {'team': 'Crystal Palace', 'year': '2020/2021', 'position': 14, 'points': 44, 'gd': -25},
    {'team': 'Southampton', 'year': '2020/2021', 'position': 15, 'points': 43, 'gd': -21},
    {'team': 'Brighton & Hove Albion', 'year': '2020/2021', 'position': 16, 'points': 41, 'gd': -6},
    {'team': 'Burnley', 'year': '2020/2021', 'position': 17, 'points': 39, 'gd': -22},
    {'team': 'Fulham', 'year': '2020/2021', 'position': 18, 'points': 28, 'gd': -26},
    {'team': 'West Bromwich Albion', 'year': '2020/2021', 'position': 19, 'points': 26, 'gd': -41},
    {'team': 'Sheffield United', 'year': '2020/2021', 'position': 20, 'points': 23, 'gd': -43},
    
    # 2021/2022 - City pip Liverpool by 1 point
    {'team': 'Manchester City', 'year': '2021/2022', 'position': 1, 'points': 93, 'gd': 73},
    {'team': 'Liverpool', 'year': '2021/2022', 'position': 2, 'points': 92, 'gd': 68},
    {'team': 'Chelsea', 'year': '2021/2022', 'position': 3, 'points': 74, 'gd': 43},
    {'team': 'Tottenham Hotspur', 'year': '2021/2022', 'position': 4, 'points': 71, 'gd': 29},
    {'team': 'Arsenal', 'year': '2021/2022', 'position': 5, 'points': 69, 'gd': 13},
    {'team': 'Manchester United', 'year': '2021/2022', 'position': 6, 'points': 58, 'gd': 0},
    {'team': 'West Ham United', 'year': '2021/2022', 'position': 7, 'points': 56, 'gd': 9},
    {'team': 'Leicester City', 'year': '2021/2022', 'position': 8, 'points': 52, 'gd': -6},
    {'team': 'Brighton & Hove Albion', 'year': '2021/2022', 'position': 9, 'points': 51, 'gd': -2},
    {'team': 'Wolverhampton', 'year': '2021/2022', 'position': 10, 'points': 51, 'gd': -5},
    {'team': 'Newcastle United', 'year': '2021/2022', 'position': 11, 'points': 49, 'gd': -18},
    {'team': 'Crystal Palace', 'year': '2021/2022', 'position': 12, 'points': 48, 'gd': 4},
    {'team': 'Brentford', 'year': '2021/2022', 'position': 13, 'points': 46, 'gd': -8},
    {'team': 'Aston Villa', 'year': '2021/2022', 'position': 14, 'points': 45, 'gd': -2},
    {'team': 'Southampton', 'year': '2021/2022', 'position': 15, 'points': 40, 'gd': -24},
    {'team': 'Everton', 'year': '2021/2022', 'position': 16, 'points': 39, 'gd': -23},
    {'team': 'Leeds United', 'year': '2021/2022', 'position': 17, 'points': 38, 'gd': -37},
    {'team': 'Burnley', 'year': '2021/2022', 'position': 18, 'points': 35, 'gd': -19},
    {'team': 'Watford', 'year': '2021/2022', 'position': 19, 'points': 23, 'gd': -43},
    {'team': 'Norwich City', 'year': '2021/2022', 'position': 20, 'points': 22, 'gd': -61},
    
    # 2022/2023 - City's treble-winning season
    {'team': 'Manchester City', 'year': '2022/2023', 'position': 1, 'points': 89, 'gd': 61},
    {'team': 'Arsenal', 'year': '2022/2023', 'position': 2, 'points': 84, 'gd': 45},
    {'team': 'Manchester United', 'year': '2022/2023', 'position': 3, 'points': 75, 'gd': 15},
    {'team': 'Newcastle United', 'year': '2022/2023', 'position': 4, 'points': 71, 'gd': 35},
    {'team': 'Liverpool', 'year': '2022/2023', 'position': 5, 'points': 67, 'gd': 28},
    {'team': 'Brighton & Hove Albion', 'year': '2022/2023', 'position': 6, 'points': 62, 'gd': 19},
    {'team': 'Aston Villa', 'year': '2022/2023', 'position': 7, 'points': 61, 'gd': 5},
    {'team': 'Tottenham Hotspur', 'year': '2022/2023', 'position': 8, 'points': 60, 'gd': 7},
    {'team': 'Brentford', 'year': '2022/2023', 'position': 9, 'points': 59, 'gd': 8},
    {'team': 'Fulham', 'year': '2022/2023', 'position': 10, 'points': 52, 'gd': -2},
    {'team': 'Crystal Palace', 'year': '2022/2023', 'position': 11, 'points': 45, 'gd': -4},
    {'team': 'Chelsea', 'year': '2022/2023', 'position': 12, 'points': 44, 'gd': -9},
    {'team': 'Wolverhampton', 'year': '2022/2023', 'position': 13, 'points': 41, 'gd': -27},
    {'team': 'West Ham United', 'year': '2022/2023', 'position': 14, 'points': 40, 'gd': -13},
    {'team': 'AFC Bournemouth', 'year': '2022/2023', 'position': 15, 'points': 39, 'gd': -26},
    {'team': 'Nottingham Forest', 'year': '2022/2023', 'position': 16, 'points': 38, 'gd': -30},
    {'team': 'Everton', 'year': '2022/2023', 'position': 17, 'points': 36, 'gd': -23},
    {'team': 'Leicester City', 'year': '2022/2023', 'position': 18, 'points': 34, 'gd': -17},
    {'team': 'Leeds United', 'year': '2022/2023', 'position': 19, 'points': 31, 'gd': -30},
    {'team': 'Southampton', 'year': '2022/2023', 'position': 20, 'points': 25, 'gd': -37},
    
    # 2023/2024 - City's 4th consecutive title
    {'team': 'Manchester City', 'year': '2023/2024', 'position': 1, 'points': 91, 'gd': 62},
    {'team': 'Arsenal', 'year': '2023/2024', 'position': 2, 'points': 89, 'gd': 62},
    {'team': 'Liverpool', 'year': '2023/2024', 'position': 3, 'points': 82, 'gd': 45},
    {'team': 'Aston Villa', 'year': '2023/2024', 'position': 4, 'points': 68, 'gd': 15},
    {'team': 'Tottenham Hotspur', 'year': '2023/2024', 'position': 5, 'points': 66, 'gd': 13},
    {'team': 'Chelsea', 'year': '2023/2024', 'position': 6, 'points': 63, 'gd': 14},
    {'team': 'Newcastle United', 'year': '2023/2024', 'position': 7, 'points': 60, 'gd': 23},
    {'team': 'Manchester United', 'year': '2023/2024', 'position': 8, 'points': 60, 'gd': -1},
    {'team': 'West Ham United', 'year': '2023/2024', 'position': 9, 'points': 52, 'gd': -14},
    {'team': 'Crystal Palace', 'year': '2023/2024', 'position': 10, 'points': 49, 'gd': -1},
    {'team': 'Brighton & Hove Albion', 'year': '2023/2024', 'position': 11, 'points': 48, 'gd': -7},
    {'team': 'AFC Bournemouth', 'year': '2023/2024', 'position': 12, 'points': 48, 'gd': -6},
    {'team': 'Fulham', 'year': '2023/2024', 'position': 13, 'points': 47, 'gd': -6},
    {'team': 'Wolverhampton', 'year': '2023/2024', 'position': 14, 'points': 46, 'gd': -12},
    {'team': 'Everton', 'year': '2023/2024', 'position': 15, 'points': 40, 'gd': -13},
    {'team': 'Brentford', 'year': '2023/2024', 'position': 16, 'points': 39, 'gd': -14},
    {'team': 'Nottingham Forest', 'year': '2023/2024', 'position': 17, 'points': 32, 'gd': -18},
    {'team': 'Luton Town', 'year': '2023/2024', 'position': 18, 'points': 26, 'gd': -33},
    {'team': 'Burnley', 'year': '2023/2024', 'position': 19, 'points': 24, 'gd': -37},
    {'team': 'Sheffield United', 'year': '2023/2024', 'position': 20, 'points': 16, 'gd': -69},
    
    # 2024/2025 - UPDATE WITH FINAL STANDINGS WHEN AVAILABLE
    {'team': 'Liverpool', 'year': '2024/2025', 'position': 1, 'points': 82, 'gd': 50},
    {'team': 'Arsenal', 'year': '2024/2025', 'position': 2, 'points': 68, 'gd': 35},
    {'team': 'Nottingham Forest', 'year': '2024/2025', 'position': 3, 'points': 67, 'gd': 15},
    {'team': 'Chelsea', 'year': '2024/2025', 'position': 4, 'points': 60, 'gd': 18},
    {'team': 'Newcastle United', 'year': '2024/2025', 'position': 5, 'points': 60, 'gd': 16},
    {'team': 'Manchester City', 'year': '2024/2025', 'position': 6, 'points': 58, 'gd': 20},
    {'team': 'AFC Bournemouth', 'year': '2024/2025', 'position': 7, 'points': 57, 'gd': 10},
    {'team': 'Aston Villa', 'year': '2024/2025', 'position': 8, 'points': 55, 'gd': 3},
    {'team': 'Fulham', 'year': '2024/2025', 'position': 9, 'points': 52, 'gd': 1},
    {'team': 'Brighton & Hove Albion', 'year': '2024/2025', 'position': 10, 'points': 50, 'gd': 3},
    {'team': 'Brentford', 'year': '2024/2025', 'position': 11, 'points': 47, 'gd': -3},
    {'team': 'Manchester United', 'year': '2024/2025', 'position': 12, 'points': 46, 'gd': -5},
    {'team': 'Tottenham Hotspur', 'year': '2024/2025', 'position': 13, 'points': 45, 'gd': -2},
    {'team': 'West Ham United', 'year': '2024/2025', 'position': 14, 'points': 41, 'gd': -15},
    {'team': 'Crystal Palace', 'year': '2024/2025', 'position': 15, 'points': 40, 'gd': -10},
    {'team': 'Everton', 'year': '2024/2025', 'position': 16, 'points': 37, 'gd': -14},
    {'team': 'Wolverhampton', 'year': '2024/2025', 'position': 17, 'points': 35, 'gd': -20},
    {'team': 'Ipswich Town', 'year': '2024/2025', 'position': 18, 'points': 32, 'gd': -24},
    {'team': 'Leicester City', 'year': '2024/2025', 'position': 19, 'points': 31, 'gd': -28},
    {'team': 'Southampton', 'year': '2024/2025', 'position': 20, 'points': 6, 'gd': -53}
]

epl_df = pd.DataFrame(epl_data)
print(f'✅ Loaded EPL standings: {len(epl_df)} team-season records')
print(f'   Seasons: {epl_df["year"].nunique()}')
epl_df.head()

---

## 3. Merge Datasets

We need to:
1. **Standardize team names** - Different sources use different naming conventions
2. **Merge on team + year** - Each row represents one team's single season
3. **Add Big 6 flag** - For later statistical analysis

In [ ]:
# Standardize team names (VAR data -> EPL data format)
name_map = {
    'Spurs': 'Tottenham Hotspur',
    'Man United': 'Manchester United',
    'Man City': 'Manchester City',
    'Newcastle': 'Newcastle United',
    'Wolves': 'Wolverhampton',
    'West Ham': 'West Ham United',
    'Leeds': 'Leeds United',
    'West Brom': 'West Bromwich Albion',
    "Nott'm Forest": 'Nottingham Forest',
    'Ipswich': 'Ipswich Town'
}
var_df['team_name'] = var_df['team_name'].replace(name_map)

# Merge datasets
merged_df = var_df.merge(
    epl_df, 
    left_on=['team_name', 'year'], 
    right_on=['team', 'year'], 
    how='inner'
)
merged_df = merged_df.drop(columns=['team'])

# Define Big 6 clubs
BIG_6 = ['Arsenal', 'Chelsea', 'Liverpool', 'Manchester City', 
         'Manchester United', 'Tottenham Hotspur']
merged_df['is_big_6'] = merged_df['team_name'].isin(BIG_6)

print(f'✅ Merged dataset: {len(merged_df)} records')
print(f'   Match rate: {len(merged_df)}/{len(var_df)} VAR records ({100*len(merged_df)/len(var_df):.1f}%)')
merged_df.head()

---

## 4. Big 6 Bias Analysis

### Research Question
> Do the traditional "Big 6" clubs (Arsenal, Chelsea, Liverpool, Man City, Man United, Spurs) receive more favorable VAR treatment than other clubs?

### Methodology
- **Test**: Welch's t-test (unequal variances assumed)
- **Null Hypothesis**: No difference in mean VAR net score between Big 6 and other clubs
- **Significance Level**: α = 0.05

In [ ]:
# Split data by Big 6 status
big6_data = merged_df[merged_df['is_big_6']]
other_data = merged_df[~merged_df['is_big_6']]

print('=' * 60)
print('🏟️  BIG 6 VS OTHER CLUBS - STATISTICAL ANALYSIS')
print('=' * 60)

# Summary statistics
print('\n📊 Summary Statistics:')
print(f'   Big 6 clubs:  n={len(big6_data)}, mean={big6_data["net_score"].mean():+.2f}, std={big6_data["net_score"].std():.2f}')
print(f'   Other clubs:  n={len(other_data)}, mean={other_data["net_score"].mean():+.2f}, std={other_data["net_score"].std():.2f}')

# Welch's t-test
t_stat, p_val = ttest_ind(big6_data['net_score'], other_data['net_score'], equal_var=False)

print('\n📈 Welch\'s t-test Results:')
print(f'   t-statistic: {t_stat:.4f}')
print(f'   p-value:     {p_val:.4f}')

if p_val < 0.05:
    print('\n   ⚠️  RESULT: Significant difference detected (p < 0.05)')
    print('   → Evidence suggests VAR treatment differs between Big 6 and other clubs')
else:
    print('\n   ✅ RESULT: No significant difference (p ≥ 0.05)')
    print('   → No evidence of systematic Big 6 bias in VAR decisions')

In [ ]:
# Visualization: Box plot comparison
fig, ax = plt.subplots(figsize=(10, 6))

data_to_plot = [big6_data['net_score'], other_data['net_score']]
bp = ax.boxplot(data_to_plot, labels=['Big 6', 'Other Clubs'], patch_artist=True)

# Styling
bp['boxes'][0].set_facecolor('#e74c3c')
bp['boxes'][1].set_facecolor('#3498db')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.7, label='Neutral (0)')

# Labels
ax.set_ylabel('VAR Net Score', fontsize=12)
ax.set_title('VAR Net Score Distribution: Big 6 vs Other Clubs', fontsize=14, fontweight='bold')

# Add sample sizes
ax.text(1, ax.get_ylim()[0], f'n={len(big6_data)}', ha='center', va='top', fontsize=10)
ax.text(2, ax.get_ylim()[0], f'n={len(other_data)}', ha='center', va='top', fontsize=10)

plt.tight_layout()
plt.savefig('images/var_big6_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved to images/var_big6_boxplot.png')

---

## 5. VAR vs League Performance Correlation

### Research Question
> Does a team's VAR net score correlate with their final league performance?

### Methodology
- **Pearson correlation**: Measures linear relationship (assumes normal distribution)
- **Spearman correlation**: Rank-based (better for league position)

### Interpretation Guide
| r value | Strength |
|---------|----------|
| 0.0 - 0.3 | Weak |
| 0.3 - 0.6 | Moderate |
| 0.6 - 1.0 | Strong |

In [ ]:
# Calculate correlations
r_points, p_points = pearsonr(merged_df['net_score'], merged_df['points'])
r_position, p_position = spearmanr(merged_df['net_score'], merged_df['position'])

print('=' * 60)
print('📈 CORRELATION ANALYSIS: VAR vs LEAGUE PERFORMANCE')
print('=' * 60)

print('\n1️⃣  VAR Net Score vs Final Points:')
print(f'    Pearson r = {r_points:+.3f}  (p = {p_points:.4f})')
strength = 'weak' if abs(r_points) < 0.3 else 'moderate' if abs(r_points) < 0.6 else 'strong'
print(f'    Interpretation: {strength.upper()} correlation')

print('\n2️⃣  VAR Net Score vs League Position:')
print(f'    Spearman r = {r_position:+.3f}  (p = {p_position:.4f})')
print('    Note: Negative = higher VAR score → better position (lower number)')

In [ ]:
# Visualization: Scatter plot with regression
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: VAR vs Points
ax = axes[0]
sns.regplot(data=merged_df, x='net_score', y='points', ax=ax,
            scatter_kws={'alpha': 0.6, 's': 50},
            line_kws={'color': 'red', 'lw': 2})
ax.set_xlabel('VAR Net Score', fontsize=11)
ax.set_ylabel('Final Points', fontsize=11)
ax.set_title(f'VAR Net Score vs Final Points\n(r = {r_points:+.3f})', fontsize=12, fontweight='bold')

# Plot 2: VAR vs Position
ax = axes[1]
sns.regplot(data=merged_df, x='net_score', y='position', ax=ax,
            scatter_kws={'alpha': 0.6, 's': 50},
            line_kws={'color': 'red', 'lw': 2})
ax.invert_yaxis()  # Lower position = better
ax.set_xlabel('VAR Net Score', fontsize=11)
ax.set_ylabel('Final Position (1 = Best)', fontsize=11)
ax.set_title(f'VAR Net Score vs League Position\n(r = {r_position:+.3f})', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('images/var_correlation_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved to images/var_correlation_scatter.png')

---

## 6. Key Findings Summary

This section summarizes the main findings from our analysis.

In [ ]:
print('=' * 60)
print('📋 KEY FINDINGS SUMMARY')
print('=' * 60)

# Finding 1: Big 6 bias
t_stat, p_val = ttest_ind(big6_data['net_score'], other_data['net_score'], equal_var=False)
bias_result = 'YES - Significant difference' if p_val < 0.05 else 'NO - No significant difference'
print(f'\n1️⃣  Big 6 Bias in VAR Decisions:')
print(f'    Result: {bias_result}')
print(f'    p-value: {p_val:.4f}')

# Finding 2: Correlation
r, p = pearsonr(merged_df['net_score'], merged_df['points'])
strength = 'WEAK' if abs(r) < 0.3 else 'MODERATE' if abs(r) < 0.6 else 'STRONG'
print(f'\n2️⃣  VAR-Points Correlation:')
print(f'    Correlation: r = {r:+.3f} ({strength})')
print(f'    Statistical significance: {"YES" if p < 0.05 else "NO"} (p = {p:.4f})')

# Finding 3: Extreme cases
most_benefited = merged_df.loc[merged_df['net_score'].idxmax()]
most_hurt = merged_df.loc[merged_df['net_score'].idxmin()]
print(f'\n3️⃣  Most Benefited Team-Season:')
print(f'    {most_benefited["team_name"]} ({most_benefited["year"]})')
print(f'    VAR Net Score: +{most_benefited["net_score"]:.0f}')

print(f'\n4️⃣  Most Disadvantaged Team-Season:')
print(f'    {most_hurt["team_name"]} ({most_hurt["year"]})')
print(f'    VAR Net Score: {most_hurt["net_score"]:.0f}')

print('\n' + '=' * 60)

---

## 7. Export Results

Save processed data for use in other analyses or reporting.

In [ ]:
# Export merged dataset
merged_df.to_csv('data/var_epl_merged.csv', index=False)
print('💾 Exported: data/var_epl_merged.csv')

print('\n✅ Analysis complete!')